In [2]:
import pandas as pd
import numpy as np



# **Extracting Links from the txt file**

In [1]:
links=[]
with open('all_ links_for_global_military_data.txt', 'r') as f:
    linkss = f.readlines()
    for  i in linkss:
        links.append(i.strip()[1:])


# **Extracting the html contents from the links using BeautifulSoup**

Checking the html content of first link

In [3]:
import requests
from bs4 import BeautifulSoup


response= requests.get(links[0])
soup =BeautifulSoup(response.content,'html.parser')



In [ ]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en-US">
 <head>
  <meta charset="utf-8"/>
  <title>
   Total Population by Country (2025)
  </title>
  <link href="https://www.globalfirepower.com/total-population-by-country.php" rel="canonical"/>
  <link href="https://www.globalfirepower.com/imgs/design/logo.png" rel="image_src"/>
  <style type="text/css">
   @font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/latin-ext/100/normal.woff2);unicode-range:U+0100-02AF,U+0304,U+0308,U+0329,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20CF,U+2113,U+2C60-2C7F,U+A720-A7FF;font-display:swap;}@font-face {font-family:IBM Plex Sans;font-style:normal;font-weight:100;src:url(/cf-fonts/s/ibm-plex-sans/5.0.18/latin/100/normal.woff2);unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+0304,U+0308,U+0329,U+2000-206F,U+2074,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD;font-display:swap;}@font-face {font-family:IBM Plex San

In [4]:
countries=[]


In [5]:
links.remove("https://www.globalfirepower.com/capital-cities-by-total-population.php")

Data(dict) is created by  extracting the value for each country (Value and Country names are extracted using their divs from the html )and the metric name is directly derived from the link

In [6]:
import requests
from bs4 import BeautifulSoup
import time

headers = {"User-Agent": "Mozilla/5.0"}
data = {}

for url in links:

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    metric=url.split("/")[-1].split(".")[0]
    rows = soup.select("div.picTrans.recordsetContainer")

    for row in rows:
        country = row.select_one(
            "div.countryNameContainer span.textWhite"
        )


        value = row.select_one(
            "div.valueContainer span span"
        )

        if not country or not value:
            continue

        cname = country.text.strip()
        val = value.text.strip()
        if cname not in data:
            data[cname] = {}
            countries.append(cname)

        data[cname][metric] = val.replace(",", "")




Handling the Capital Case seperately because the country name is not mentioned only in this capital_cities_population

In [7]:
i=0
response = requests.get("https://www.globalfirepower.com/capital-cities-by-total-population.php", headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

rows = soup.select("div.picTrans.recordsetContainer")
for row in  rows:
        capital = row.select_one(
            "div.countryNameContainer span.textWhite"
        )
        value = row.select_one(
            "div.valueContainer span span"
        )

        data[countries[i]]['Capital']=capital.text.strip()

        data[countries[i]]['Captal Population']=value.text.strip()
        i+=1


# **Converting it into the Dataframe and then to csv**

In [8]:


df = pd.DataFrame.from_dict(data, orient="index")
df.index.name = "Country"
df.reset_index(inplace=True)
df.to_csv("countries_data.csv", index=False)



In [9]:
 df.shape

(145, 56)

In [ ]:
df.head()

,Country,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,aircraft-total,aircraft-total-fighters,...,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage,Capital,Captal Population
0,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,6654000000000 \t\t\t\t\...,4827000000 \t\t\t\t\t\t...,5313000000 \t\t\t\t\t\t...,143197000000 \t\t\t\t\t...,9596960 \t\t\t\t\t\t\r\...,14500 \t\t\t\t\t\t\r\n\...,22457 \t\t\t\t\t\t\r\n\...,27700 \t\t\t\t\t\t\r\n\...,Tokyo,"37,274,000"
1,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,1381000000000 \t\t\t\t\...,985671000 \t\t\t\t\t\t\...,1200000000 \t\t\t\t\t\t...,111052000000 \t\t\t\t\t...,3287263 \t\t\t\t\t\t\r\...,7000 \t\t\t\t\t\t\r\n\t...,13888 \t\t\t\t\t\t\r\n\...,14500 \t\t\t\t\t\t\r\n\...,New Delhi,"32,066,000"
2,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,13402000000000 \t\t\t\t...,548849000 \t\t\t\t\t\t\...,476044000 \t\t\t\t\t\t\...,248941000000 \t\t\t\t\t...,9833517 \t\t\t\t\t\t\r\...,19924 \t\t\t\t\t\t\r\n\...,12002 \t\t\t\t\t\t\r\n\...,41009 \t\t\t\t\t\t\r\n\...,Dhaka,"22,478,000"
3,Indonesia,281562465,137965608,114595923,4786562,400000,400000,250000,459,41,...,1408000000000 \t\t\t\t\...,659357000 \t\t\t\t\t\t\...,202283000 \t\t\t\t\t\t\...,34869000000 \t\t\t\t\t\...,1904569 \t\t\t\t\t\t\r\...,54716 \t\t\t\t\t\t\r\n\...,2958 \t\t\t\t\t\t\r\n\t...,21579 \t\t\t\t\t\t\r\n\...,Mexico City,"22,281,000"
4,Pakistan,252363571,108516336,85803614,4794908,654000,550000,500000,1399,328,...,592219000000 \t\t\t\t\t...,12712000 \t\t\t\t\t\t\r...,34027000 \t\t\t\t\t\t\r...,3064000000 \t\t\t\t\t\t...,796095 \t\t\t\t\t\t\r\n...,1046 \t\t\t\t\t\t\r\n\t...,7257 \t\t\t\t\t\t\r\n\t...,0 \t\t\t\t\t\t\r\n\t\t\...,Cairo,"21,750,000"


In [23]:
url="https://www.globalfirepower.com/powerindex.php"

In [24]:
data={}

In [26]:


url = "https://www.globalfirepower.com/countries-listing.php"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers, timeout=15)
soup = BeautifulSoup(response.text, "html.parser")

rows = soup.select("div.picTrans.recordsetContainer")

data = []

for row in rows:
    rank = row.select_one("div.rankNumContainer span")
    country = row.select_one("div.countryName div.longFormName")

    if not rank or not country:
        continue

    data.append({
        "country": country.text.strip(),
        "power_index_rank": int(rank.text.strip())
    })

df_rank = pd.DataFrame(data)

print(df_rank.head())


         country  power_index_rank
0  United States                 1
1         Russia                 2
2          China                 3
3          India                 4
4    South Korea                 5


In [27]:
data

[{'country': 'United States', 'power_index_rank': 1},
 {'country': 'Russia', 'power_index_rank': 2},
 {'country': 'China', 'power_index_rank': 3},
 {'country': 'India', 'power_index_rank': 4},
 {'country': 'South Korea', 'power_index_rank': 5},
 {'country': 'United Kingdom', 'power_index_rank': 6},
 {'country': 'France', 'power_index_rank': 7},
 {'country': 'Japan', 'power_index_rank': 8},
 {'country': 'Turkiye', 'power_index_rank': 9},
 {'country': 'Italy', 'power_index_rank': 10},
 {'country': 'Brazil', 'power_index_rank': 11},
 {'country': 'Pakistan', 'power_index_rank': 12},
 {'country': 'Indonesia', 'power_index_rank': 13},
 {'country': 'Germany', 'power_index_rank': 14},
 {'country': 'Israel', 'power_index_rank': 15},
 {'country': 'Iran', 'power_index_rank': 16},
 {'country': 'Spain', 'power_index_rank': 17},
 {'country': 'Australia', 'power_index_rank': 18},
 {'country': 'Egypt', 'power_index_rank': 19},
 {'country': 'Ukraine', 'power_index_rank': 20},
 {'country': 'Poland', 'po

In [29]:


df_rank= pd.DataFrame(data)


In [34]:
df_rank.head()
df_rank.rename(columns={'country':'Country'},inplace=True)

In [35]:
df_rank.head()

,Country,power_index_rank
0,United States,1
1,Russia,2
2,China,3
3,India,4
4,South Korea,5


In [31]:
df.head()

,Country,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,aircraft-total,aircraft-total-fighters,...,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage,Capital,Captal Population
0,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,6654000000000 \t\t\t\t\...,4827000000 \t\t\t\t\t\t...,5313000000 \t\t\t\t\t\t...,143197000000 \t\t\t\t\t...,9596960 \t\t\t\t\t\t\r\...,14500 \t\t\t\t\t\t\r\n\...,22457 \t\t\t\t\t\t\r\n\...,27700 \t\t\t\t\t\t\r\n\...,Tokyo,"37,274,000"
1,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,1381000000000 \t\t\t\t\...,985671000 \t\t\t\t\t\t\...,1200000000 \t\t\t\t\t\t...,111052000000 \t\t\t\t\t...,3287263 \t\t\t\t\t\t\r\...,7000 \t\t\t\t\t\t\r\n\t...,13888 \t\t\t\t\t\t\r\n\...,14500 \t\t\t\t\t\t\r\n\...,New Delhi,"32,066,000"
2,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,13402000000000 \t\t\t\t...,548849000 \t\t\t\t\t\t\...,476044000 \t\t\t\t\t\t\...,248941000000 \t\t\t\t\t...,9833517 \t\t\t\t\t\t\r\...,19924 \t\t\t\t\t\t\r\n\...,12002 \t\t\t\t\t\t\r\n\...,41009 \t\t\t\t\t\t\r\n\...,Dhaka,"22,478,000"
3,Indonesia,281562465,137965608,114595923,4786562,400000,400000,250000,459,41,...,1408000000000 \t\t\t\t\...,659357000 \t\t\t\t\t\t\...,202283000 \t\t\t\t\t\t\...,34869000000 \t\t\t\t\t\...,1904569 \t\t\t\t\t\t\r\...,54716 \t\t\t\t\t\t\r\n\...,2958 \t\t\t\t\t\t\r\n\t...,21579 \t\t\t\t\t\t\r\n\...,Mexico City,"22,281,000"
4,Pakistan,252363571,108516336,85803614,4794908,654000,550000,500000,1399,328,...,592219000000 \t\t\t\t\t...,12712000 \t\t\t\t\t\t\r...,34027000 \t\t\t\t\t\t\r...,3064000000 \t\t\t\t\t\t...,796095 \t\t\t\t\t\t\r\n...,1046 \t\t\t\t\t\t\r\n\t...,7257 \t\t\t\t\t\t\r\n\t...,0 \t\t\t\t\t\t\r\n\t\t\...,Cairo,"21,750,000"


In [38]:
df=df.merge(df_rank,on='Country')

In [39]:
df.head()

,Country,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,aircraft-total,aircraft-total-fighters,...,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage,Capital,Captal Population,power_index_rank
0,China,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,1212,...,4827000000 \t\t\t\t\t\t...,5313000000 \t\t\t\t\t\t...,143197000000 \t\t\t\t\t...,9596960 \t\t\t\t\t\t\r\...,14500 \t\t\t\t\t\t\r\n\...,22457 \t\t\t\t\t\t\r\n\...,27700 \t\t\t\t\t\t\r\n\...,Tokyo,"37,274,000",3
1,India,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,513,...,985671000 \t\t\t\t\t\t\...,1200000000 \t\t\t\t\t\t...,111052000000 \t\t\t\t\t...,3287263 \t\t\t\t\t\t\r\...,7000 \t\t\t\t\t\t\r\n\t...,13888 \t\t\t\t\t\t\r\n\...,14500 \t\t\t\t\t\t\r\n\...,New Delhi,"32,066,000",4
2,United States,341963408,150463900,124816644,4445524,1328000,799500,0,13043,1790,...,548849000 \t\t\t\t\t\t\...,476044000 \t\t\t\t\t\t\...,248941000000 \t\t\t\t\t...,9833517 \t\t\t\t\t\t\r\...,19924 \t\t\t\t\t\t\r\n\...,12002 \t\t\t\t\t\t\r\n\...,41009 \t\t\t\t\t\t\r\n\...,Dhaka,"22,478,000",1
3,Indonesia,281562465,137965608,114595923,4786562,400000,400000,250000,459,41,...,659357000 \t\t\t\t\t\t\...,202283000 \t\t\t\t\t\t\...,34869000000 \t\t\t\t\t\...,1904569 \t\t\t\t\t\t\r\...,54716 \t\t\t\t\t\t\r\n\...,2958 \t\t\t\t\t\t\r\n\t...,21579 \t\t\t\t\t\t\r\n\...,Mexico City,"22,281,000",13
4,Pakistan,252363571,108516336,85803614,4794908,654000,550000,500000,1399,328,...,12712000 \t\t\t\t\t\t\r...,34027000 \t\t\t\t\t\t\r...,3064000000 \t\t\t\t\t\t...,796095 \t\t\t\t\t\t\r\n...,1046 \t\t\t\t\t\t\r\n\t...,7257 \t\t\t\t\t\t\r\n\t...,0 \t\t\t\t\t\t\r\n\t\t\...,Cairo,"21,750,000",12


In [40]:
df.to_csv("countries_data.csv", index=False)